In [ ]:
#!pip install -q -U langchain==0.2.16 langchain-google-genai langchain-community faiss-cpu pypdf google-generativeai langchain-core

In [ ]:
import os

GOOGLE_API_KEY = "ABq9GA49GN03BAUSBgnHSHHMQ9A1KgBjLM12u3yNDG"
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document

In [ ]:
from google.colab import files
uploaded = files.upload()
file_name = list(uploaded.keys())[0]
loader = PyPDFLoader(file_name)
raw_docs = loader.load()

Saving DM Lab.pdf to DM Lab (1).pdf


In [ ]:
def split_documents(documents):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=100
    )
    return splitter.split_documents(documents)

chunks = split_documents(raw_docs)
print(f"Split into {len(chunks)} chunks")

# Preview one chunk
print("\n--- Sample Chunk ---")
print(chunks[0].page_content)

Split into 52 chunks

--- Sample Chunk ---
EX:1   
1.  Aim  
To  study  the  positive  and  negative  impacts  and  ethical  implications  of  using  social  media  for  
political
 
advertising.
 
 
2.  Description  
Social  media  platforms  like  Facebook,  Twitter  (X),  and  Instagram  are  widely  used  for  political  
advertising
 
to
 
reach
 
a
 
large
 
audience
 
quickly.
 
Political
 
parties
 
and
 
candidates
 
use
 
these
 
platforms
 
to
 
share
 
campaigns,
 
promote
 
ideologies,
 
and
 
influence
 
voters.


In [ ]:
print("Creating embeddings...")

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

vectorstore = FAISS.from_documents(chunks, embeddings)
print(f"FAISS vector store created with {vectorstore.index.ntotal} vectors!")

Creating embeddings...
FAISS vector store created with 52 vectors!


In [ ]:
llm = ChatGoogleGenerativeAI(
    model="models/gemini-2.5-flash",
    temperature=0.3
)

prompt = ChatPromptTemplate.from_template("""
You are a helpful AI Knowledge Assistant.
Answer the question ONLY from the context provided below.
If the answer is not in the context, say: "I couldn't find that information in the document."
Keep your answer clear and concise.

Context:
{context}

Question: {input}

Answer:
""")

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

document_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, document_chain)

print("RAG pipeline ready!")

RAG pipeline ready!


In [ ]:
conversation_memory = []

def add_to_memory(role, message):
    conversation_memory.append({"role": role, "message": message})

def get_memory_context():
    """Returns last 4 exchanges as extra context"""
    if not conversation_memory:
        return ""
    recent = conversation_memory[-8:]
    formatted = "\n".join([f"{m['role'].upper()}: {m['message']}" for m in recent])
    return f"\n\nPrevious conversation:\n{formatted}"


def qa_tool(query):
    """Answer a question from the document"""
    full_query = query + get_memory_context()
    result = rag_chain.invoke({"input": full_query})
    return result["answer"]

def summarize_tool(query):
    """Summarize relevant content"""
    result = rag_chain.invoke({"input": f"Give a brief summary of: {query}"})
    return result["answer"]

def explain_tool(query):
    """Explain something in simple terms"""
    result = rag_chain.invoke({"input": f"Explain clearly in simple terms: {query}"})
    return result["answer"]


def agent(query):
    q = query.lower()

    if "summarize" in q or "summary" in q:
        response = summarize_tool(query)
        tool_used = "Summarize"
    elif "explain" in q or "what is" in q or "what are" in q:
        response = explain_tool(query)
        tool_used = "Explain"
    else:
        response = qa_tool(query)
        tool_used = "Q&A"

    add_to_memory("User", query)
    add_to_memory("AI", response)

    return response, tool_used

print("Memory + Agent ready!")

Memory + Agent ready!


In [ ]:
print("AI Knowledge Assistant - Ready!")
print("=" * 50)
print("Tips:")
print("  - Ask anything about the document")
print("  - Say 'summarize ...' to summarize a topic")
print("  - Say 'explain ...' to get a simple explanation")
print("  - Type 'history' to see past conversation")
print("  - Type 'exit' to quit")
print("=" * 50)

while True:
    query = input("\nYou: ").strip()

    if not query:
        print("Please type something!")
        continue

    if query.lower() == "exit":
        print("\nGoodbye! Session ended.")
        break

    if query.lower() == "history":
        if not conversation_memory:
            print("\nNo history yet.")
        else:
            print("\nConversation History:")
            print("-" * 40)
            for m in conversation_memory:
                print(f"{m['role']}: {m['message']}")
            print("-" * 40)
        continue

    try:
        response, tool = agent(query)
        print(f"\nAI [{tool}]:\n{response}")
    except Exception as e:
        print(f"\nError: {str(e)}")
        print("Check your API key and internet connection.")

AI Knowledge Assistant - Ready!
Tips:
  - Ask anything about the document
  - Say 'summarize ...' to summarize a topic
  - Say 'explain ...' to get a simple explanation
  - Type 'history' to see past conversation
  - Type 'exit' to quit

You: explain the postive impact 

AI [Explain]:
The positive impacts of using social media for political advertising include:
*   **Wide Reach:** Political messages can instantly reach millions of people.
*   **Cost-Effective:** It is cheaper than traditional media like TV and newspapers.
*   **Targeted Advertising:** Ads can be customized based on user demographics and interests.
*   **Increased Engagement:** Voters can interact directly through comments, likes, and shares.
*   **Awareness:** It helps educate people about political issues and candidates.

You: summarize the file

AI [Summarize]:
The document describes two main studies:
1.  An aim to study the positive and negative impacts and ethical implications of using social media for political ad